In [2]:
from google.colab import drive
drive.mount('/content/drive')


import pandas as pd
import numpy as np
import os


Mounted at /content/drive


In [3]:
DATASET_FOLDER = '/content/drive/MyDrive/ML_Challenge_2025/dataset/'

train = pd.read_csv(os.path.join(DATASET_FOLDER, 'train.csv'))
test = pd.read_csv(os.path.join(DATASET_FOLDER, 'test.csv'))

print(train.shape, test.shape)
train.head()


(75000, 4) (75000, 3)


,sample_id,catalog_content,image_link,price
0,33127,"Item Name: La Victoria Green Taco Sauce Mild, ...",https://m.media-amazon.com/images/I/51mo8htwTH...,4.89
1,198967,"Item Name: Salerno Cookies, The Original Butte...",https://m.media-amazon.com/images/I/71YtriIHAA...,13.12
2,261251,"Item Name: Bear Creek Hearty Soup Bowl, Creamy...",https://m.media-amazon.com/images/I/51+PFEe-w-...,1.97
3,55858,Item Name: Judee’s Blue Cheese Powder 11.25 oz...,https://m.media-amazon.com/images/I/41mu0HAToD...,30.34
4,292686,"Item Name: kedem Sherry Cooking Wine, 12.7 Oun...",https://m.media-amazon.com/images/I/41sA037+Qv...,66.49


In [4]:
import re

def extract_features(text):
    val_match = re.search(r'Value:\s*([\d.]+)', str(text))
    unit_match = re.search(r'Unit:\s*([\w\s]+)', str(text))
    pack_match = re.search(r'Pack of\s*(\d+)', str(text), re.IGNORECASE)

    value = float(val_match.group(1)) if val_match else np.nan
    unit = unit_match.group(1).strip() if unit_match else "Unknown"
    pack = int(pack_match.group(1)) if pack_match else 1
    return value, unit, pack

train[['Value', 'Unit', 'Pack']] = train['catalog_content'].apply(lambda x: pd.Series(extract_features(x)))
test[['Value', 'Unit', 'Pack']] = test['catalog_content'].apply(lambda x: pd.Series(extract_features(x)))

train[['Value', 'Unit', 'Pack']].head()


,Value,Unit,Pack
0,72.00,Fl Oz,6
1,32.00,Ounce,4
2,11.40,Ounce,6
3,11.25,Ounce,1
4,12.00,Count,1


In [6]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
le.fit(list(train['Unit'].values) + list(test['Unit'].values))  # fit on both train + test

train['Unit'] = le.transform(train['Unit'])
test['Unit'] = le.transform(test['Unit'])


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
X_text_train = tfidf.fit_transform(train['catalog_content'])
X_text_test = tfidf.transform(test['catalog_content'])

print(X_text_train.shape)


(75000, 5000)


In [9]:
from scipy.sparse import hstack

# Numeric features
X_num_train = np.array(train[['Value', 'Pack', 'Unit']].fillna(0))
X_num_test = np.array(test[['Value', 'Pack', 'Unit']].fillna(0))

# Combine sparse + numeric
X_train = hstack([X_text_train, X_num_train])
X_test = hstack([X_text_test, X_num_test])
y_train = train['price'].values


In [10]:
!pip install lightgbm


In [19]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

model = lgb.LGBMRegressor(
    objective='regression_l1',
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=64,
    random_state=42
)

model.fit(X_tr, y_tr,
          eval_set=[(X_val, y_val)],
          callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=100)])

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 4.079762 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 515240
[LightGBM] [Info] Number of data points in the train set: 60000, number of used features: 4984
[LightGBM] [Info] Start training from score 14.090000
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1000]	valid_0's l1: 11.7162


LGBMRegressor(learning_rate=0.05, n_estimators=1000, num_leaves=64,
              objective='regression_l1', random_state=42)

In [20]:
def smape(y_true, y_pred):
    return 100/len(y_true) * np.sum(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred)))

y_val_pred = model.predict(X_val)
print("Validation SMAPE:", smape(y_val, y_val_pred))


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Validation SMAPE: 52.99229484364188


In [17]:
preds = model.predict(X_test)
preds = np.clip(preds, 0, None)  # ensure positive prices

output = pd.DataFrame({
    'sample_id': test['sample_id'],
    'price': preds
})

# Save output in the correct format
output_path = '/content/drive/MyDrive/ML_Challenge_2025/test_out.csv'
output.to_csv(output_path, index=False)

print(f"✅ Saved predictions to: {output_path}")
output.head()


/usr/local/lib/python3.12/dist-packages/lightgbm/basic.py:1238: UserWarning: Converting data to scipy sparse matrix.
  _log_warning("Converting data to scipy sparse matrix.")


✅ Saved predictions to: /content/drive/MyDrive/ML_Challenge_2025/test_out.csv


,sample_id,price
0,100179,16.543446
1,245611,21.574068
2,146263,22.447795
3,95658,10.268591
4,36806,28.298265


In [22]:
# Run this first cell
!pip install -q lightgbm sentence-transformers datasets transformers --upgrade

# If you only want TF-IDF (faster), sentence-transformers is optional


In [23]:
from google.colab import drive
drive.mount('/content/drive')

# Edit this path to your folder that contains dataset/
DRIVE_ROOT = '/content/drive/MyDrive/ML_Challenge_2025'
DATASET_FOLDER = f'{DRIVE_ROOT}/dataset'
OUTPUT_FOLDER = f'{DRIVE_ROOT}/outputs'
MODEL_FOLDER = f'{DRIVE_ROOT}/models'

import os
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
os.makedirs(MODEL_FOLDER, exist_ok=True)

print("DATASET_FOLDER:", DATASET_FOLDER)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DATASET_FOLDER: /content/drive/MyDrive/ML_Challenge_2025/dataset


In [24]:
import pandas as pd, numpy as np
train = pd.read_csv(os.path.join(DATASET_FOLDER, 'train.csv'))
test = pd.read_csv(os.path.join(DATASET_FOLDER, 'test.csv'))
sample_test = pd.read_csv(os.path.join(DATASET_FOLDER, 'sample_test.csv')) if os.path.exists(os.path.join(DATASET_FOLDER, 'sample_test.csv')) else None

print("train:", train.shape, "test:", test.shape)
train.head(2)


train: (75000, 4) test: (75000, 3)


,sample_id,catalog_content,image_link,price
0,33127,"Item Name: La Victoria Green Taco Sauce Mild, ...",https://m.media-amazon.com/images/I/51mo8htwTH...,4.89
1,198967,"Item Name: Salerno Cookies, The Original Butte...",https://m.media-amazon.com/images/I/71YtriIHAA...,13.12


In [25]:
import re

def extract_value_unit_pack(text):
    s = str(text)
    # Value: 72.0  or Value:72
    v = None
    m = re.search(r'Value:\s*([\d,.]+)', s, flags=re.IGNORECASE)
    if m:
        v = float(m.group(1).replace(',', ''))
    # Unit: Ounce or Unit: Fl Oz
    unit = None
    m2 = re.search(r'Unit:\s*([A-Za-z0-9 %\.\-/]+)', s, flags=re.IGNORECASE)
    if m2:
        unit = m2.group(1).strip()
    # Pack of 6 or (Pack of 6) or Pack of 4
    pack = 1
    m3 = re.search(r'Pack(?:\s*of)?\s*(\d+)', s, flags=re.IGNORECASE)
    if m3:
        try:
            pack = int(m3.group(1))
        except:
            pack = 1
    # Also look for "case of X" or "pack of X" variations
    m4 = re.search(r'(?:case|pack|packs|pcs|pieces)\s*(?:of)?\s*(\d+)', s, flags=re.IGNORECASE)
    if m4:
        try:
            pack = int(m4.group(1))
        except:
            pass
    return v, unit if unit else "Unknown", pack

for df in [train, test]:
    df[['Value', 'Unit', 'Pack']] = df['catalog_content'].apply(lambda x: pd.Series(extract_value_unit_pack(x)))

print(train[['Value','Unit','Pack']].head(8))


    Value   Unit  Pack
0   72.00  Fl Oz     6
1   32.00  Ounce     4
2   11.40  Ounce     6
3   11.25  Ounce     1
4   12.00  Count     1
5    6.25  ounce     1
6  180.00  Ounce     6
7    1.00  Count     1


In [26]:
# conversions: grams -> oz, kg->oz, lb -> oz, fl oz -> oz (volume treated as oz), ounce stays
def unit_to_base_qty(value, unit):
    if value is None or pd.isna(value):
        return np.nan
    u = str(unit).lower()
    v = float(value)
    # grams
    if re.search(r'gram|g\b', u):
        return v / 28.3495231  # g -> oz
    if re.search(r'kg\b|kilogram', u):
        return (v * 1000) / 28.3495231
    if re.search(r'lb\b|pound', u):
        return v * 16.0  # lb -> oz
    if re.search(r'fl oz|floz|fl\. oz|fluid ounce', u):
        return v  # treat fl oz as oz base
    if re.search(r'ounce|oz\b', u):
        return v
    # counts (pieces, count) - treat as count unit, keep as-is
    if re.search(r'count|piece|pcs|packets|tablet|capsule|caps', u):
        return v  # count-like unit, interpret as count
    # milliliter -> convert to fl oz approx (1 fl oz = 29.5735 ml)
    if re.search(r'ml\b|milliliter', u):
        return v / 29.5735296  # ml -> fl oz ~ oz
    # fallback: Unknown or complex units -> return value as-is (model may learn)
    return v

for df in [train, test]:
    df['Value_base_oz'] = df.apply(lambda r: unit_to_base_qty(r['Value'], r['Unit']), axis=1)
    # total quantity = value * pack
    df['Total_Qty_oz'] = df['Value_base_oz'] * df['Pack']

# quick look
print(train[['Value','Unit','Pack','Value_base_oz','Total_Qty_oz']].head(8))


    Value   Unit  Pack  Value_base_oz  Total_Qty_oz
0   72.00  Fl Oz     6          72.00        432.00
1   32.00  Ounce     4          32.00        128.00
2   11.40  Ounce     6          11.40         68.40
3   11.25  Ounce     1          11.25         11.25
4   12.00  Count     1          12.00         12.00
5    6.25  ounce     1           6.25          6.25
6  180.00  Ounce     6         180.00       1080.00
7    1.00  Count     1           1.00          1.00


In [27]:
# Fill missing numeric with median or 0
for col in ['Value', 'Value_base_oz', 'Total_Qty_oz']:
    med = train[col].median(skipna=True)
    train[col] = train[col].fillna(med)
    test[col] = test[col].fillna(med)

# Create price per unit (for train only) and log target
train['price_per_totalqty'] = train['price'] / (train['Total_Qty_oz'].replace(0, np.nan))
train['price_per_totalqty'] = train['price_per_totalqty'].fillna(train['price'])  # fallback

# log transform target (helps regression)
train['log_price'] = np.log1p(train['price'])

# quick stats
print("price stats:", train['price'].describe())
print("log_price stats:", train['log_price'].describe())


price stats: count    75000.000000
mean        23.647654
std         33.376932
min          0.130000
25%          6.795000
50%         14.000000
75%         28.625000
max       2796.000000
Name: price, dtype: float64
log_price stats: count    75000.000000
mean         2.739217
std          0.942032
min          0.122218
25%          2.053483
50%          2.708050
75%          3.388619
max          7.936303
Name: log_price, dtype: float64


In [28]:
# Create a safe mapping from union of train+test units
units = list(pd.Series(list(train['Unit'].astype(str)) + list(test['Unit'].astype(str))).unique())
unit_mapping = {u: i for i, u in enumerate(sorted(units))}
# map, unseen -> -1 (shouldn't happen because mapping includes train+test union)
train['Unit_enc'] = train['Unit'].map(unit_mapping).fillna(-1).astype(int)
test['Unit_enc'] = test['Unit'].map(unit_mapping).fillna(-1).astype(int)

# Show some
print("Unique units mapped:", len(unit_mapping))
train[['Unit','Unit_enc']].head(6)


Unique units mapped: 142


,Unit,Unit_enc
0,Fl Oz,44
1,Ounce,70
2,Ounce,70
3,Ounce,70
4,Count,35
5,ounce,126


In [29]:
USE_SBERT = False   # <-- set True to use Sentence-BERT (slower but better)

if USE_SBERT:
    from sentence_transformers import SentenceTransformer
    print("Loading SBERT (this may take time)...")
    sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
    # encode text into dense vectors
    X_text_train = sbert_model.encode(train['catalog_content'].tolist(), show_progress_bar=True)
    X_text_test = sbert_model.encode(test['catalog_content'].tolist(), show_progress_bar=True)
    # X_text_* are numpy arrays
    print("SBERT shapes:", X_text_train.shape, X_text_test.shape)
else:
    # TF-IDF sparse features
    from sklearn.feature_extraction.text import TfidfVectorizer
    tfidf = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1,2))
    X_text_train = tfidf.fit_transform(train['catalog_content'].fillna(''))
    X_text_test = tfidf.transform(test['catalog_content'].fillna(''))
    print("TF-IDF shapes:", X_text_train.shape, X_text_test.shape)


TF-IDF shapes: (75000, 5000) (75000, 5000)


In [30]:
from scipy.sparse import hstack, csr_matrix

# numeric features to use
num_cols = ['Value', 'Value_base_oz', 'Total_Qty_oz', 'Pack', 'Unit_enc']
X_num_train = train[num_cols].astype(float).values
X_num_test = test[num_cols].astype(float).values

if USE_SBERT:
    # X_text_* are dense numpy arrays
    X_train = np.hstack([X_text_train, X_num_train])
    X_test = np.hstack([X_text_test, X_num_test])
else:
    # X_text_* are sparse matrix
    X_train = hstack([X_text_train, csr_matrix(X_num_train)])
    X_test = hstack([X_text_test, csr_matrix(X_num_test)])

y = train['log_price'].values  # train target is log price
print("Combined shapes:", getattr(X_train,'shape', None), getattr(X_test,'shape', None))


Combined shapes: (75000, 5005) (75000, 5005)


In [32]:
from sklearn.model_selection import train_test_split
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y, test_size=0.2, random_state=42)

import lightgbm as lgb

# LightGBM can accept sparse matrices
lgb_train = lgb.Dataset(X_tr, label=y_tr)
lgb_val = lgb.Dataset(X_val, label=y_val)

params = {
    'objective': 'regression',
    'metric': 'l2',        # training metric; we'll evaluate SMAPE separately
    'learning_rate': 0.05,
    'num_leaves': 64,
    'verbosity': -1,
    'seed': 42,
    'n_jobs': -1
}

print("Training LightGBM...")
bst = lgb.train(
    params,
    lgb_train,
    num_boost_round=2000,
    valid_sets=[lgb_train, lgb_val],
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=100)]
)

Training LightGBM...
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1019]	training's l2: 0.201844	valid_1's l2: 0.46712


In [33]:
def smape(y_true, y_pred):
    # expects raw prices (not logs). Returns decimal (e.g., 0.18 => 18%)
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    # avoid division by zero
    mask = denom == 0
    denom[mask] = 1.0
    return np.mean(np.abs(y_pred - y_true) / denom) * 100.0

# predict on validation (remember we trained on log price)
y_val_pred_log = bst.predict(X_val, num_iteration=bst.best_iteration)
y_val_pred = np.expm1(y_val_pred_log)   # back to price
y_val_true = np.expm1(y_val)            # back to price

val_smape = smape(y_val_true, y_val_pred)
print(f"Validation SMAPE: {val_smape:.6f}%")


Validation SMAPE: 51.725474%


In [34]:
# predict log prices on X_test then convert back
y_test_pred_log = bst.predict(X_test, num_iteration=bst.best_iteration)
y_test_pred = np.expm1(y_test_pred_log)
# Ensure positivity
y_test_pred = np.clip(y_test_pred, 0, None)

out_df = pd.DataFrame({'sample_id': test['sample_id'], 'price': y_test_pred})
out_path = os.path.join(OUTPUT_FOLDER, 'test_out.csv')
out_df.to_csv(out_path, index=False)
print("Saved predictions to:", out_path)
out_df.head()


Saved predictions to: /content/drive/MyDrive/ML_Challenge_2025/outputs/test_out.csv


,sample_id,price
0,100179,13.105413
1,245611,14.305327
2,146263,14.925338
3,95658,12.278369
4,36806,33.812820


In [35]:
# Optional: Download images. Use the provided download_images util if available.
# If you don't have src/utils.py, use simple requests with retries.

import requests, time
from pathlib import Path

IMAGES_FOLDER = f'{DRIVE_ROOT}/images_test'
Path(IMAGES_FOLDER).mkdir(parents=True, exist_ok=True)

def download_url(url, fname, retries=3, timeout=10):
    for attempt in range(retries):
        try:
            r = requests.get(url, timeout=timeout)
            if r.status_code == 200:
                with open(fname, 'wb') as f:
                    f.write(r.content)
                return True
        except Exception as e:
            time.sleep(1 + attempt)
    return False

# Example: download first 200 images (adjust as needed)
sample = test['image_link'].dropna().unique()[:200]
for i, url in enumerate(sample):
    fname = os.path.join(IMAGES_FOLDER, f'{i}.jpg')
    ok = download_url(url, fname)
    if not ok:
        print("Failed:", url)
print("Downloaded up to 200 sample images to", IMAGES_FOLDER)


Downloaded up to 200 sample images to /content/drive/MyDrive/ML_Challenge_2025/images_test


In [37]:
import lightgbm as lgb
from google.colab import drive
import os


# Define a path to save the model
model_path = '/content/drive/MyDrive/ML_Challenge_2025/price_prediction_lgbm_model.txt'

# Save the trained LightGBM model
bst.save_model(model_path)

print(f"Model saved to: {model_path}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model saved to: /content/drive/MyDrive/ML_Challenge_2025/price_prediction_lgbm_model.txt


In [39]:
import pickle

# Save TF-IDF vectorizer (already fitted)
tfidf_path = f'{MODEL_FOLDER}/tfidf_vectorizer.pkl'
with open(tfidf_path, 'wb') as f:
    pickle.dump(tfidf, f)

print("TF-IDF vectorizer saved at:", tfidf_path)


TF-IDF vectorizer saved at: /content/drive/MyDrive/ML_Challenge_2025/models/tfidf_vectorizer.pkl
